# 03 — Automatic differentiation and PDE residuals

This notebook isolates the differentiable computational graph behind a PINN: first derivatives, second derivatives, residual construction, parameter gradients, and common failure modes.

In [ ]:
import torch
from torch import nn
from pinn import MLP, heat_residual, heat_exact_solution
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
alpha=0.1


## 1. Verify autograd on an analytic function

For $f=x^2t+\sin(t)$, the derivatives are $f_x=2xt$, $f_t=x^2+\cos(t)$, and $f_{xx}=2t$. This isolates PyTorch differentiation before introducing a neural network.

In [ ]:
xt=torch.tensor([[0.4,0.7],[-0.3,0.2]],requires_grad=True)
def f(z):
    x,t=z[:,0:1],z[:,1:2]
    return x**2*t+torch.sin(t)
y=f(xt)
dy=torch.autograd.grad(y,xt,torch.ones_like(y),create_graph=True)[0]
fx,ft=dy[:,0:1],dy[:,1:2]
fxx=torch.autograd.grad(fx,xt,torch.ones_like(fx),create_graph=True)[0][:,0:1]
print('fx error',float((fx-2*xt[:,0:1]*xt[:,1:2]).abs().max()))
print('ft error',float((ft-(xt[:,0:1]**2+torch.cos(xt[:,1:2]))).abs().max()))
print('fxx error',float((fxx-2*xt[:,1:2]).abs().max()))


## 2. Differentiate a network output with respect to coordinates

In [ ]:
model=MLP(hidden_dim=16,hidden_layers=2)
coords=torch.cat([torch.rand(64,1)*2-1,torch.rand(64,1)],1).requires_grad_(True)
u=model(coords)
du=torch.autograd.grad(u,coords,torch.ones_like(u),create_graph=True)[0]
print('u:',u.shape,'du:',du.shape,'graph:',du.requires_grad)


## 3. Second derivative and residual

In [ ]:
u_x,u_t=du[:,0:1],du[:,1:2]
d2=torch.autograd.grad(u_x,coords,torch.ones_like(u_x),create_graph=True)[0]
u_xx=d2[:,0:1]
residual=u_t-alpha*u_xx
print('u_xx:',u_xx.shape,'residual:',residual.shape)


## 4. Package residual and gradient diagnostics

A PDE loss can legitimately give `None` for a parameter that the particular derivative operator eliminates. Therefore, the robust test is not that every parameter has a gradient; it is that the residual/loss has a valid finite gradient path into the model.

In [ ]:
model.zero_grad(set_to_none=True)
r=heat_residual(model,coords,alpha)
loss=r.square().mean()
loss.backward()
for name,p in model.named_parameters():
    print(f'{name:30s}', 'has_grad=',p.grad is not None, 'norm=',None if p.grad is None else float(p.grad.norm()))


## 5. Exact solution residual identity

The analytical heat-equation solution must satisfy $u_t-\alpha u_{xx}=0$ up to floating-point roundoff.

In [ ]:
class Exact(nn.Module):
    def forward(self,z): return heat_exact_solution(z[:,0:1],z[:,1:2],alpha)
test=torch.cat([torch.rand(1000,1)*2-1,torch.rand(1000,1)],1).requires_grad_(True)
r=heat_residual(Exact(),test,alpha)
print('max |r|=',float(r.abs().max()))
print('RMSE=',float(torch.sqrt(torch.mean(r.square()))))


## 6. Input/model validation failures

The production implementation should reject malformed coordinate batches and non-scalar field outputs early.

In [ ]:
for bad in [torch.rand(8),torch.rand(8,3)]:
    try: heat_residual(model,bad,alpha)
    except Exception as exc: print(type(exc).__name__,exc)
class TwoOutputs(nn.Module):
    def forward(self,z): return torch.cat([z[:,0:1],z[:,1:2]],1)
try: heat_residual(TwoOutputs(),torch.rand(8,2),alpha)
except Exception as exc: print(type(exc).__name__,exc)
